# Part 1 - Get rid of data leakage

### We will try several unsupervised classic ML approaches, implemented in OpenCV, to get rid of leaked images in test dataset

### Our goal is to achieve F1 >= 0.81 with 15 minutes timeout. My config: M3 Max 4E + 12P

### Set the target folder path, containing train/test sets in their folders, and leakage info file

In [2]:
import os
from glob import glob

import cv2
from tqdm import tqdm


DATA_DIR = "public_data"
TEST_DIR = os.path.join(DATA_DIR, "test")
TRAIN_DIR = os.path.join(DATA_DIR, "train")
LEAKAGE_FILE_PATH = os.path.join(DATA_DIR, "leakage_files.txt")

In [3]:
gt_leaked = []
with open(LEAKAGE_FILE_PATH) as f:
    for l in f.readlines():
        gt_leaked.append(l.strip("\n"))

## ORB

### Preprocessing: make sure we will not do any redunant calculations

In [ ]:
# define the only ORB and matcher objects
orb = cv2.ORB_create()
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)

In [41]:
def orb_preprocess(img_path):
    img = cv2.imread(img_path, 0)

    _, des1 = orb.detectAndCompute(img, None)
    return des1

In [ ]:
# precalculating all nessessary values
train_path_orb = {}
for p in tqdm(glob(f"{TRAIN_DIR}/*")):
    train_path_orb[p] = orb_preprocess(p)

test_path_orb = {}
for p in tqdm(glob(f"{TEST_DIR}/*")):
    test_path_orb[p] = orb_preprocess(p)

100%|██████████| 136/136 [00:03<00:00, 40.19it/s]


### Images comparison (test vs train): getting leaked imgs

In [ ]:
# This number suits `public_data` to have nice F1
MATCHES_THRESHOLD = 70

In [43]:
def compare_imgs(orb_test, orb_train):
    matches = bf.knnMatch(orb_test, orb_train, k=2)

    good = []
    for m, n in matches:
        if m.distance < 0.75 * n.distance:
            good.append([m])

    if len(good) > MATCHES_THRESHOLD:
        return True
    
    return

In [ ]:
# bruteforce all the test vs train images to get leaked ones
pred_leaked = []
for k, v in tqdm(test_path_orb.items()):
    for _v in train_path_orb.values():
        res = compare_imgs(v, _v)
        if res:
            break

    if res:
        pred_leaked.append(k)

print(f"LEAKED AMOUNT: {len(pred_leaked)}")

100%|██████████| 136/136 [00:15<00:00,  8.63it/s]

LEAKED AMOUNT: 56


### Metrics: count are we good or not

In [45]:
pred_set = set([os.path.basename(el) for el in pred_leaked])
gt_set = set(gt_leaked)

TP = len(pred_set & gt_set)  # intersection
FP = len(pred_set - gt_set)  # in pred but not in gt
FN = len(gt_set - pred_set)  # in gt but not in pred

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Precision: {precision:.2f}")
print(f"Recall:    {recall:.2f}")
print(f"F1-score:  {f1:.2f}")

Precision: 0.84
Recall:    0.80
F1-score:  0.82


#### So we already got required results! But let's try something else

## AKAZE

### Preprocessing: make sure we will not do any redunant calculations

In [ ]:
# define the only AKAZE and matcher objects
akaze = cv2.AKAZE_create()
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)

In [ ]:
def akaze_preprocess(img_path):
    img = cv2.imread(img_path, 0)

    _, des1 = akaze.detectAndCompute(img, None)
    return des1

In [ ]:
# precalculating all nessessary values
train_path_akaze = {}
for p in tqdm(glob(f"{TRAIN_DIR}/*")):
    train_path_akaze[p] = akaze_preprocess(p)

test_path_akaze = {}
for p in tqdm(glob(f"{TEST_DIR}/*")):
    test_path_akaze[p] = akaze_preprocess(p)

100%|██████████| 136/136 [00:10<00:00, 12.59it/s]


### Images comparison (test vs train): getting leaked imgs

In [66]:
# This number suits `public_data` to have nice F1
MATCHES_THRESHOLD = 140

In [ ]:
def compare_imgs(akaze_test, akaze_train):
    matches = bf.knnMatch(akaze_test, akaze_train, k=2)

    good = []
    for m, n in matches:
        if m.distance < 0.75 * n.distance:
            good.append([m])

    if len(good) > MATCHES_THRESHOLD:
        return True
    
    return

In [68]:
# bruteforce all the test vs train images to get leaked ones
pred_leaked = []
for k, v in tqdm(test_path_akaze.items()):
    for _v in train_path_akaze.values():
        res = compare_imgs(v, _v)
        if res:
            break

    if res:
        pred_leaked.append(k)

print(f"LEAKED AMOUNT: {len(pred_leaked)}")

100%|██████████| 136/136 [12:22<00:00,  5.46s/it]

LEAKED AMOUNT: 66


### Metrics: count are we good or not

In [70]:
pred_set = set([os.path.basename(el) for el in pred_leaked])
gt_set = set(gt_leaked)

TP = len(pred_set & gt_set)  # intersection
FP = len(pred_set - gt_set)  # in pred but not in gt
FN = len(gt_set - pred_set)  # in gt but not in pred

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Precision: {precision:.2f}")
print(f"Recall:    {recall:.2f}")
print(f"F1-score:  {f1:.2f}")

Precision: 0.83
Recall:    0.93
F1-score:  0.88


#### AKAZE gave us significantly better metrics (probably could achieve F1 >= 0.9), but a price of really slow perfomance

## SIFT

### Preprocessing: make sure we will not do any redunant calculations

In [20]:
INDEX_PARAMS = dict(algorithm = 1, trees = 3)
SEARCH_PARAMS = dict(checks = 10)

# define the only SIFT and matcher objects
sift = cv2.SIFT_create()
flann = cv2.FlannBasedMatcher(INDEX_PARAMS, SEARCH_PARAMS)

In [21]:
def sift_preprocess(img_path):
    img = cv2.imread(img_path, 0)

    _, des1 = sift.detectAndCompute(img, None)
    return des1

In [22]:
# precalculating all nessessary values
train_path_sift = {}
for p in tqdm(glob(f"{TRAIN_DIR}/*")):
    train_path_sift[p] = sift_preprocess(p)

test_path_sift = {}
for p in tqdm(glob(f"{TEST_DIR}/*")):
    test_path_sift[p] = sift_preprocess(p)

100%|██████████| 136/136 [00:17<00:00,  7.59it/s]


### Images comparison (test vs train): getting leaked imgs

In [40]:
# This number suits `public_data` to have nice F1
MATCHES_THRESHOLD = 625

In [41]:
def compare_imgs(sift_test, sift_train):
    matches = flann.knnMatch(sift_test, sift_train, k=2)

    good = []
    for m, n in matches:
        if m.distance < 0.75 * n.distance:
            good.append([m])

    if len(good) > MATCHES_THRESHOLD:
        return True
    
    return

In [42]:
# bruteforce all the test vs train images to get leaked ones
pred_leaked = []
for k, v in tqdm(test_path_sift.items()):
    for _v in train_path_sift.values():
        res = compare_imgs(v, _v)
        if res:
            break

    if res:
        pred_leaked.append(k)

print(f"LEAKED AMOUNT: {len(pred_leaked)}")

100%|██████████| 136/136 [21:09<00:00,  9.34s/it]

LEAKED AMOUNT: 55


### Metrics: count are we good or not

In [43]:
pred_set = set([os.path.basename(el) for el in pred_leaked])
gt_set = set(gt_leaked)

TP = len(pred_set & gt_set)  # intersection
FP = len(pred_set - gt_set)  # in pred but not in gt
FN = len(gt_set - pred_set)  # in gt but not in pred

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Precision: {precision:.2f}")
print(f"Recall:    {recall:.2f}")
print(f"F1-score:  {f1:.2f}")

Precision: 0.89
Recall:    0.83
F1-score:  0.86


#### SIFT gave us worse result than AZARE, and also too slow (>15 min)